In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE  # For handling class imbalance

# Load dataset
dataset_path = "a.csv"
data = pd.read_csv(dataset_path)

# Drop missing values
data.dropna(inplace=True)

# Features and target
features = ['age', 'gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'result', 'relation', 'A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 
            'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'm_m']
target = 'Class/ASD'

data[target] = data[target].apply(lambda x: 1 if x == 'YES' else 0)  # Convert target to binary

# Apply one-hot encoding for categorical variables
data_encoded = pd.get_dummies(data[features])

# Normalize numerical features
scaler = MinMaxScaler()
numerical_features = ['age', 'result', 'm_m']
data_encoded[numerical_features] = scaler.fit_transform(data_encoded[numerical_features])

# Split dataset into train/test sets
X_train, X_test, y_train, y_test = train_test_split(data_encoded, data[target], test_size=0.2, random_state=42)

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)


# SVM Model
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(X_train, y_train)

# Predict on test data
y_pred_svm = svm_model.predict(X_test)

# Accuracy
svm_accuracy = accuracy_score(y_test, y_pred_svm)
print(f"SVM Model Accuracy: {svm_accuracy * 100:.2f}%")

# ROC Curve for SVM
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_model.decision_function(X_test))
roc_auc_svm = auc(fpr_svm, tpr_svm)


SVM Model Accuracy: 95.00%


In [ ]:
import pickle

# Save the trained model
model_filename = "autism_model.pkl"
with open(model_filename, "wb") as file:
    pickle.dump(svm_model, file)

print(f"Model saved as {model_filename}")
